# Predição de resultados do Brasileirão Série A

Classificação do resultado de partidas do Campeonato Brasileiro Série A (2018–2023) a partir de atributos pré-jogo.

Dois cenários são avaliados:

- **com empates** — três classes (`casa`, `empate`, `fora`);
- **sem empates** — duas classes (`fora` vs `nao_fora`), mantido apenas para comparação com os trabalhos relacionados.

O cenário principal é o multiclasse com empates. Todo o código vive em `football_prediction/`.

## 1. Imports e configuração do projeto

In [1]:
import sys
from dataclasses import replace
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "brasileirao_serie_a_2018_2023_v3.csv").exists():
            return candidate
    raise FileNotFoundError("Raiz do projeto nao encontrada a partir do diretorio atual.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from football_prediction.baselines import compare_with_baselines
from football_prediction.evaluation import (
    classification_report_text,
    confusion_matrix_df,
    print_experiment_summary,
    run_experiment,
)
from football_prediction.experiments import (
    compare_models_by_scenario,
    make_config,
)
from football_prediction.interpretability import (
    explain_one_with_lime,
    explain_with_shap,
    prepare_interpretability_data,
)
from football_prediction.thresholds import (
    evaluate_thresholds,
    precision_at_recall,
    precision_recall_frontier,
)
from football_prediction.tuning import tune_all

/Users/matheus.soares/Downloads/tcc/football-ml-prediction/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Experimento principal

O split 70/30 é reservado para o teste final. A avaliação de desenvolvimento usa validação
cruzada repetida (5 folds x 5 repeticoes = 25 medições) sobre o conjunto de treino: com apenas
614 jogos de teste, um split único tem erro-padrão de ~2 pontos percentuais, maior que a
diferença entre os modelos comparados.

In [2]:
config = make_config(
    classifier_name="xgboost",
    include_draws=True,
    random_state=42,
    use_sample_weight=True,
)
config = replace(
    config,
    target=replace(
        config.target,
        dataset_path=PROJECT_ROOT / "brasileirao_serie_a_2018_2023_v3.csv",
    ),
)

result = run_experiment(config)
print_experiment_summary(result)

Modelo: xgboost
Dataset: brasileirao_serie_a_2018_2023_v3.csv
Target original: vencedor
Target modelado: vencedor_modelado
Split: 70% treino / 30% teste final
Cenário: com empates
Empates removidos: 0
Colunas descartadas: data
Features usadas: 35
Exemplos de treino: 1432
Exemplos de teste final: 614
Avaliações no treino: 25 (5 folds x 5 repetições)
Classes: ['casa', 'empate', 'fora']

Validação cruzada repetida no treino (média ± desvio entre folds):
Acurácia média: 0.4775 ± 0.0279
Balanced accuracy média: 0.4560 ± 0.0264
Macro-F1 médio: 0.4441 ± 0.0270
Acurácia out-of-fold agregada: 0.4735
Balanced accuracy out-of-fold agregada: 0.4511
Macro-F1 out-of-fold agregado: 0.4398

Teste final reservado:
Acurácia: 0.4446
Balanced accuracy: 0.4202
Macro-F1: 0.4134


## 3. Linhas de base

Sem uma referência explícita, uma acurácia isolada não significa nada. São três referências:

1. **classe majoritária** — sempre `casa`. É o piso da acurácia (47,5 % nesta base).
2. **argmax das odds** — a aposta mais provável segundo o mercado, com o overround removido.
3. **odds ajustadas pelo prior** — a mesma probabilidade dividida pela frequência das classes,
   que é a regra de decisão de custo balanceado. É o piso da acurácia balanceada e do macro-F1.

A comparação com (2) e (3) responde à pergunta que os trabalhos relacionados não fazem:
o modelo aprende algo além do que as casas de apostas já precificaram?

In [3]:
baselines_df = compare_with_baselines(result)
baselines_df

,abordagem,acuracia,balanced_accuracy,macro_f1,fora_precision,fora_recall,fora_f1
0,Modelo: xgboost,0.444625,0.420172,0.413398,0.350515,0.453333,0.395349
1,Baseline: classe majoritária,0.475570,0.333333,0.214864,0.000000,0.000000,0.000000
2,Baseline: argmax das odds,0.485342,0.397458,0.335068,0.378571,0.353333,0.365517
3,Baseline: odds ajustadas pelo prior,0.444625,0.443660,0.403147,0.332258,0.686667,0.447826


## 4. Comparação dos modelos por cenário

In [4]:
comparison_summary_df, hyperparameters_df, experiment_results = compare_models_by_scenario(config)

print("Comparação por cenário")
display(comparison_summary_df)

print("Hiperparâmetros usados em cada cenário (originados da busca da seção 8)")
display(hyperparameters_df)

Comparação por cenário


,cenario,classificador,classes,empates_removidos,treino,teste,cv_acuracia_media,cv_acuracia_std,cv_balanced_accuracy_media,cv_macro_f1_medio,...,fora_precision,fora_recall,fora_f1,fora_suporte,verdadeiros_fora,nao_fora_precision,nao_fora_recall,nao_fora_f1,nao_fora_suporte,verdadeiros_nao_fora
0,com empates,naive_bayes,"casa, empate, fora",0,1432,614,0.455310,0.021631,0.431106,0.423576,...,0.326425,0.420000,0.367347,150.0,63,NaN,NaN,NaN,NaN,NaN
1,com empates,svm,"casa, empate, fora",0,1432,614,0.459095,0.025482,0.448513,0.439933,...,0.314136,0.400000,0.351906,150.0,60,NaN,NaN,NaN,NaN,NaN
2,com empates,xgboost,"casa, empate, fora",0,1432,614,0.477521,0.027901,0.455974,0.444064,...,0.350515,0.453333,0.395349,150.0,68,NaN,NaN,NaN,NaN,NaN
3,com empates,mlp_classifier,"casa, empate, fora",0,1432,614,0.468451,0.023328,0.446562,0.437039,...,0.330317,0.486667,0.393531,150.0,73,NaN,NaN,NaN,NaN,NaN
7,sem empates,mlp_classifier,"fora, nao_fora",574,1030,442,0.661553,0.028289,0.658126,0.643343,...,0.492228,0.633333,0.553936,150.0,95,0.779116,0.664384,0.717190,292.0,194.0
5,sem empates,svm,"fora, nao_fora",574,1030,442,0.671650,0.025258,0.657592,0.648102,...,0.481081,0.593333,0.531343,150.0,89,0.762646,0.671233,0.714026,292.0,196.0
6,sem empates,xgboost,"fora, nao_fora",574,1030,442,0.688932,0.021751,0.669571,0.662740,...,0.476684,0.613333,0.536443,150.0,92,0.767068,0.654110,0.706100,292.0,191.0
4,sem empates,naive_bayes,"fora, nao_fora",574,1030,442,0.654951,0.025415,0.654651,0.638490,...,0.472362,0.626667,0.538682,150.0,94,0.769547,0.640411,0.699065,292.0,187.0


Hiperparâmetros usados em cada cenário (originados da busca da seção 8)


,cenario,classificador,hiperparametros
3,com empates,mlp_classifier,"{'learning_rate': 'constant', 'hidden_layer_si..."
0,com empates,naive_bayes,{'var_smoothing': 0.046667}
1,com empates,svm,"{'kernel': 'rbf', 'C': 0.196743, 'gamma': 0.02..."
2,com empates,xgboost,"{'gamma': 4.784004, 'n_estimators': 605, 'lear..."
7,sem empates,mlp_classifier,"{'learning_rate': 'constant', 'hidden_layer_si..."
4,sem empates,naive_bayes,{'var_smoothing': 0.046667}
5,sem empates,svm,"{'kernel': 'rbf', 'C': 26.37334, 'gamma': 0.00..."
6,sem empates,xgboost,"{'gamma': 3.478922, 'n_estimators': 671, 'lear..."


## 5. Resultados do modelo principal

In [5]:
result.fold_results_df.head(10)

,repeticao,fold,exemplos_treino,exemplos_validacao,acuracia,balanced_accuracy,macro_f1
0,1,1,1145,287,0.480836,0.450092,0.441696
1,1,2,1145,287,0.491289,0.468723,0.465162
2,1,3,1146,286,0.461538,0.441632,0.428847
3,1,4,1146,286,0.451049,0.437115,0.426624
4,1,5,1146,286,0.482517,0.457598,0.433011
5,2,1,1145,287,0.484321,0.455500,0.445054
6,2,2,1145,287,0.459930,0.438343,0.425862
7,2,3,1146,286,0.503497,0.476786,0.465914
8,2,4,1146,286,0.461538,0.437535,0.430734
9,2,5,1146,286,0.524476,0.515651,0.502203


In [6]:
print("Média e desvio entre as 25 avaliações da validação cruzada repetida:")
result.fold_results_df[["acuracia", "balanced_accuracy", "macro_f1"]].agg(["mean", "std"])

Média e desvio entre as 25 avaliações da validação cruzada repetida:


,acuracia,balanced_accuracy,macro_f1
mean,0.477521,0.455974,0.444064
std,0.027901,0.026425,0.027039


In [7]:
print(classification_report_text(result))

              precision    recall  f1-score   support

        casa     0.5709    0.5514    0.5610       292
      empate     0.3188    0.2558    0.2839       172
        fora     0.3505    0.4533    0.3953       150

    accuracy                         0.4446       614
   macro avg     0.4134    0.4202    0.4134       614
weighted avg     0.4465    0.4446    0.4429       614



In [8]:
confusion_matrix_df(result)

,casa,empate,fora
casa,161,61,70
empate,72,44,56
fora,49,33,68


In [9]:
result.final_model.prediction_examples.head(10)

,ano_campeonato,mes_campeonato,rodada,time_mandante,time_visitante,estadio,PPJ_pre_jogo_mandante,PPJ_pre_jogo_visitante,xG_pre_jogo_mandante,xG_pre_jogo_visitante,...,valor_equipe_titular_mandante,valor_equipe_titular_visitante,idade_media_titular_mandante,idade_media_titular_visitante,resultado_real,vencedor_real,vencedor_previsto,prob_casa,prob_empate,prob_fora
586,2019,11,30,Chapecoense,São Paulo,Arena Condá,0.79,1.36,1.48,1.24,...,10550000,41000000,29.9,27.3,fora,fora,fora,0.148412,0.350694,0.500894
246,2018,10,31,Santos,Fluminense,Estádio Urbano Caldeira,1.73,0.87,1.69,1.45,...,39300000,6400000,26.4,25.1,casa,casa,casa,0.608613,0.226278,0.165110
1063,2021,9,20,Atlético-GO,Corinthians,Estádio Antônio Accioly,1.22,1.90,1.37,1.22,...,7230000,27050000,28.6,27.3,empate,empate,fora,0.184204,0.380330,0.435466
1127,2021,10,25,Cuiabá,São Paulo,Arena Pantanal,1.08,1.08,1.51,1.53,...,7700000,43800000,28.0,26.2,empate,empate,fora,0.162677,0.342269,0.495054
1122,2021,10,27,Flamengo,Cuiabá,Estadio Jornalista Mário Filho - Maracanã,2.00,1.42,2.14,1.12,...,66500000,6800000,28.1,29.4,empate,empate,casa,0.640959,0.211280,0.147760
866,2020,12,27,Palmeiras,Bragantino,Allianz Parque,1.83,0.77,1.83,1.64,...,64200000,20000000,25.4,25.2,casa,casa,empate,0.367116,0.386598,0.246287
1875,2023,5,5,Botafogo,Corinthians,Estádio Nilton Santos,3.00,0.00,1.67,0.86,...,23500000,47400000,30.0,27.9,casa,casa,casa,0.414369,0.368277,0.217354
1557,2023,4,1,Botafogo,São Paulo,Estádio Nilton Santos,0.00,0.00,0.00,0.00,...,20800000,28400000,29.9,29.1,casa,casa,fora,0.293485,0.342634,0.363881
132,2018,8,17,Vitória,Cruzeiro,Estádio Manoel Barradas - Barradão,1.86,1.00,1.60,1.53,...,14150000,16750000,25.7,27.3,empate,empate,fora,0.214543,0.305883,0.479574
1371,2022,7,18,Bragantino,Fortaleza,Estádio Nabi Abi Chedid,1.56,0.75,1.87,1.39,...,43200000,12350000,24.3,28.6,casa,casa,casa,0.399880,0.340884,0.259236


## 6. Limiar de decisão e fronteira precisão x revocação

A varredura de limiar mostra o custo de arriscar mais a classe `fora`. A fronteira completa
permite comparar este trabalho com um ponto de operação publicado por outro autor sem depender
da escolha de limiar de cada um — a pergunta respondida é "qual a minha precisão na revocação
em que o outro trabalho opera".

In [10]:
evaluate_thresholds(result)

,limiar_fora,acuracia,balanced_accuracy,macro_f1,fora_precision,fora_recall,fora_f1,verdadeiros_fora,falsos_fora,fora_perdidos
3,0.45,0.447883,0.414381,0.416988,0.406780,0.320000,0.358209,48,70,102
1,0.35,0.447883,0.424957,0.413915,0.359223,0.493333,0.415730,74,132,76
2,0.40,0.442997,0.411126,0.410536,0.370130,0.380000,0.375000,57,97,93
4,0.50,0.444625,0.407379,0.408994,0.451220,0.246667,0.318966,37,45,113
0,0.30,0.434853,0.425208,0.391875,0.329787,0.620000,0.430556,93,189,57
5,0.55,0.436482,0.392857,0.379688,0.465116,0.133333,0.207254,20,23,130
6,0.60,0.416938,0.364485,0.318502,0.250000,0.013333,0.025316,2,6,148
7,0.65,0.421824,0.369730,0.314867,0.000000,0.000000,0.000000,0,0,150
8,0.70,0.421824,0.369730,0.314867,0.000000,0.000000,0.000000,0,0,150


In [11]:
frontier_df = precision_recall_frontier(result)

# Izidoro (2025), Tabela 7: LightGBM otimizado por SHAP, revocação de 0,195 na classe `fora`.
IZIDORO_RECALL = 0.195
IZIDORO_PRECISION = 0.358

comparable_point = precision_at_recall(frontier_df, IZIDORO_RECALL)
print(f"Ponto de operação de Izidoro (2025): revocação {IZIDORO_RECALL:.3f}, precisão {IZIDORO_PRECISION:.3f}")
print(f"Este trabalho na mesma revocação   : revocação {comparable_point['recall']:.3f}, precisão {comparable_point['precision']:.3f}")
print(f"Limiar correspondente              : {comparable_point['limiar']:.3f}")

Ponto de operação de Izidoro (2025): revocação 0.195, precisão 0.358
Este trabalho na mesma revocação   : revocação 0.193, precisão 0.509
Limiar correspondente              : 0.532


## 7. Busca de hiperparâmetros

Os valores em `football_prediction/models.py` vêm desta busca, e não de escolha manual.
A avaliação usa somente o conjunto de treino, para que o teste final continue sendo um dado
nunca visto.

A célula abaixo é lenta (algumas centenas de ajustes). Ela reproduz os valores já gravados
no `build_model_registry`.

In [12]:
tuning_summary_df, tuning_results = tune_all(config)
tuning_summary_df

/Users/matheus.soares/Downloads/tcc/football-ml-prediction/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/matheus.soares/Downloads/tcc/football-ml-prediction/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/matheus.soares/Downloads/tcc/football-ml-prediction/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/matheus.soares/Downloads/tcc/football-ml-prediction/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: 

/Users/matheus.soares/Downloads/tcc/football-ml-prediction/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1200) reached and the optimization hasn't converged yet.
  warnings.warn(


/Users/matheus.soares/Downloads/tcc/football-ml-prediction/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1200) reached and the optimization hasn't converged yet.
  warnings.warn(


/Users/matheus.soares/Downloads/tcc/football-ml-prediction/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1200) reached and the optimization hasn't converged yet.
  warnings.warn(


/Users/matheus.soares/Downloads/tcc/football-ml-prediction/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1200) reached and the optimization hasn't converged yet.
  warnings.warn(


...

/Users/matheus.soares/Downloads/tcc/football-ml-prediction/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/matheus.soares/Downloads/tcc/football-ml-prediction/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1200) reached and the optimization hasn't converged yet.
  warnings.warn(


/Users/matheus.soares/Downloads/tcc/football-ml-prediction/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/matheus.soares/Downloads/tcc/football-ml-prediction/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/matheus.soares/Downloads/tcc/football-ml-prediction/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1200) reached and the optimization hasn't converged yet.
  warnings.warn(


/Users/matheus.soares/Downloads/tcc/football-ml-prediction/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/matheus.soares/Downloads/tcc/football-ml-prediction/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/matheus.soares/Downloads/tcc/football-ml-prediction/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1200) reached and the optimization hasn't converged yet.
  warnings.warn(


/Users/matheus.soares/Downloads/tcc/football-ml-prediction/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/matheus.soares/Downloads/tcc/football-ml-prediction/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1200) reached and the optimization hasn't converged yet.
  warnings.warn(


,cenario,classificador,metrica,melhor_score_cv,combinacoes_testadas,melhores_hiperparametros
2,com empates,xgboost,f1_macro,0.459712,60,"{'colsample_bytree': 0.9693313223519999, 'gamm..."
3,com empates,mlp_classifier,f1_macro,0.451440,40,"{'alpha': 0.9877700294007907, 'batch_size': 32..."
1,com empates,svm,f1_macro,0.445441,40,"{'C': 145.28246637516014, 'gamma': 0.000115264..."
0,com empates,naive_bayes,f1_macro,0.442083,30,{'var_smoothing': 0.028698379250004825}


## 8. Interpretabilidade com SHAP e LIME

A coluna `data` é excluída das features: com 592 valores distintos em 2 046 linhas, ela gerava
592 colunas one-hot que dominavam as explicações do LIME sem acrescentar poder preditivo.

In [13]:
interpretability_data = prepare_interpretability_data(result)
print(f"Features após pré-processamento: {len(interpretability_data['transformed_feature_names'])}")
print(f"Colunas categóricas identificadas: {len(interpretability_data['categorical_feature_indexes'])}")
print(f"Amostra de fundo SHAP: {len(interpretability_data['shap_background'])} registros do treino")
print(f"Amostra explicada SHAP: {len(interpretability_data['shap_explain_data'])} registros do teste final")

Features após pré-processamento: 194
Colunas categóricas identificadas: 166
Amostra de fundo SHAP: 200 registros do treino
Amostra explicada SHAP: 300 registros do teste final


In [14]:
shap_importance_df = explain_with_shap(result, interpretability_data)
shap_importance_df.head(15)

,feature,mean_abs_shap
0,num__odds_visitante_vence,0.105226
1,num__odds_mandante_vence,0.070857
2,num__valor_equipe_titular_mandante,0.044447
3,num__colocacao_mandante,0.039062
4,num__xG_pre_jogo_visitante,0.036834
5,num__colocacao_visitante,0.031095
6,num__odds_AM_nao,0.030334
7,num__valor_equipe_titular_visitante,0.027545
8,num__idade_media_titular_mandante,0.023249
9,num__odds_AM_sim,0.020011


In [15]:
lime_explanation_df = explain_one_with_lime(result, interpretability_data, example_index=0)
print(f"Índice explicado no teste final: {lime_explanation_df.attrs['example_index']}")
print(f"Classe real: {lime_explanation_df.attrs['real_class']}")
print(f"Classe prevista: {lime_explanation_df.attrs['predicted_class']}")
print(f"Probabilidade prevista: {lime_explanation_df.attrs['predicted_probability']:.4f}")
lime_explanation_df

Índice explicado no teste final: 0
Classe real: fora
Classe prevista: fora
Probabilidade prevista: 0.5009


,feature_condition,contribution
0,num__odds_visitante_vence <= 2.73,0.128675
1,num__odds_mandante_vence > 2.70,0.070399
2,num__colocacao_mandante > 16.00,0.042905
3,cat__estadio_Estádio Francisco Stédile=0,0.036870
4,cat__formacao_visitante_4-3-1-2=0,0.034039
5,cat__formacao_mandante_3-5-1-1=0,-0.034030
6,cat__formacao_visitante_3-2-4-1=0,-0.032724
7,num__ano_campeonato <= 2019.00,-0.029233
8,cat__estadio_Estádio Municipal Jacy Scaff=0,-0.029204
9,num__valor_equipe_titular_visitante > 31387500.00,0.028188
